In [ ]:
# Clone the BioMedKG repository containing the graph data with node features
!git clone https://github.com/hysonlab/biomedkg.git

# Navigate to the directory and list the contents to find data paths
%cd biomedkg
!ls -lh

# Note: Please check the cloned repository's README or data folder to locate
# the specific node features and graph structure files (e.g., .pt, .csv, or .pkl files)
# that you can load into your PyTorch Geometric (PyG) or DGL pipeline.

Cloning into 'biomedkg'...
remote: Enumerating objects: 796, done.
remote: Counting objects: 100% (543/543), done.
remote: Compressing objects: 100% (272/272), done.
remote: Total 796 (delta 326), reused 433 (delta 271), pack-reused 253 (from 1)
Receiving objects: 100% (796/796), 1.30 MiB | 14.74 MiB/s, done.
Resolving deltas: 100% (481/481), done.
/content/biomedkg
total 60K
drwxr-xr-x 2 root root 4.0K Oct  2 14:42 assets
drwxr-xr-x 5 root root 4.0K Oct  2 14:42 biomedkg
drwxr-xr-x 5 root root 4.0K Oct  2 14:42 configs
-rw-r--r-- 1 root root  416 Oct  2 14:42 Makefile
-rw-r--r-- 1 root root 4.4K Oct  2 14:42 ml_exp.py
-rw-r--r-- 1 root root  526 Oct  2 14:42 pyproject.toml
-rw-r--r-- 1 root root 6.7K Oct  2 14:42 README.md
drwxr-xr-x 3 root root 4.0K Oct  2 14:42 scripts
-rw-r--r-- 1 root root 1.2K Oct  2 14:42 test_dpi.py
-rw-r--r-- 1 root root 1.2K Oct  2 14:42 test_kge.py
-rw-r--r-- 1 root root 3.1K Oct  2 14:42 train_dpi.py
-rw-r--r-- 1 root root 3.5K Oct  2 14:42 train_gcl.py
-rw

In [3]:
# ==========================================
# 1. INSTALL DEPENDENCIES
# ==========================================
!pip install -q huggingface_hub torch torch-geometric pandas numpy

import os
import pandas as pd
import numpy as np
import torch
from torch_geometric.data import HeteroData
from huggingface_hub import snapshot_download
from pathlib import Path

# ==========================================
# 2. DOWNLOAD DATASET FROM HUGGING FACE
# ==========================================
print("Downloading BioMedKG dataset...")
dataset_dir = "./BioMedKG_data"

snapshot_download(
    repo_id="tienda02/BioMedKG",
    repo_type="dataset",
    local_dir=dataset_dir,
)
print(f"Dataset downloaded to {dataset_dir}!\n")

# ==========================================
# 3. INITIALIZE GRAPH & UTILS
# ==========================================
graph = HeteroData()

node_mappings = {}
node_counters = {}

def get_node_id(node_type, node_name):
    """Maps a node's string name to a continuous integer ID for graph construction."""
    if node_type not in node_mappings:
        node_mappings[node_type] = {}
        node_counters[node_type] = 0

    if node_name not in node_mappings[node_type]:
        node_mappings[node_type][node_name] = node_counters[node_type]
        node_counters[node_type] += 1

    return node_mappings[node_type][node_name]

# ==========================================
# 4. PARSE FILES AND BUILD GRAPH
# ==========================================
print("Processing dataset files...")
csv_files = list(Path(dataset_dir).rglob("*.csv"))
edge_dfs = []

for file_path in csv_files:
    try:
        df = pd.read_csv(file_path, low_memory=False)
    except Exception as e:
        continue

    edge_cols = {'x_type', 'x_name', 'relation', 'y_type', 'y_name'}
    if edge_cols.issubset(df.columns):
        edge_dfs.append(df)

# ==========================================
# 5. LOAD EDGES INTO HETERODATA
# ==========================================
if edge_dfs:
    print("\nConstructing graph edges...")
    full_edges_df = pd.concat(edge_dfs, ignore_index=True)
    grouped_edges = full_edges_df.groupby(['x_type', 'relation', 'y_type'])

    for (x_type, relation, y_type), group in grouped_edges:
        src_nodes = [get_node_id(x_type, str(name)) for name in group['x_name']]
        dst_nodes = [get_node_id(y_type, str(name)) for name in group['y_name']]

        edge_index = torch.tensor([src_nodes, dst_nodes], dtype=torch.long)
        graph[(x_type, relation, y_type)].edge_index = edge_index
        print(f"  -> Added {len(src_nodes)} edges for {(x_type, relation, y_type)}")

# ==========================================
# 6. DECLARE NODES & LOAD/INIT FEATURES
# ==========================================
print("\nInitializing nodes and features...")

# Tell PyG exactly how many nodes exist for each type
for node_type, count in node_counters.items():
    graph[node_type].num_nodes = count
    print(f"  -> Initialized {count} nodes for type: '{node_type}'")

pt_files = list(Path(dataset_dir).rglob("*.pt"))
npy_files = list(Path(dataset_dir).rglob("*.npy"))

if pt_files or npy_files:
    # Load explicit feature tensors if they exist in the repo
    for feature_file in pt_files + npy_files:
        node_type = feature_file.stem.split('_')[0]
        if node_type in graph.node_types:
            features = torch.load(feature_file) if feature_file.suffix == '.pt' else torch.tensor(np.load(feature_file), dtype=torch.float32)
            graph[node_type].x = features
            print(f"  -> Loaded features for '{node_type}': shape {features.shape}")
else:
    # BioMedKG's paper states they generate Language Model embeddings.
    # Since the raw tensors aren't in the base repo, we initialize dummy embeddings
    # (e.g., dim=768 like standard BERT/SciBERT) so the PyG graph is ready for training logic.
    print("\nNo explicit .pt/.npy files found. Initializing dummy feature tensors (dim=768) so graph is ready for GNN layers.")
    embedding_dim = 768
    for node_type in graph.node_types:
        num_nodes = graph[node_type].num_nodes
        graph[node_type].x = torch.randn(num_nodes, embedding_dim)
        print(f"  -> Assigned dummy features for '{node_type}': shape [{num_nodes}, {embedding_dim}]")

# ==========================================
# 7. VALIDATE THE GRAPH
# ==========================================
print("\n=== Graph Construction Complete ===")
print(graph)
print(f"Is graph valid? {graph.validate()}")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 20 files:   0%|          | 0/20 [00:00<?, ?it/s]

Dataset downloaded to ./BioMedKG_data!

Processing dataset files...

Constructing graph edges...
  -> Added 21447 edges for ('drug', 'DPI', 'gene/protein')

Initializing nodes and features...
  -> Initialized 7801 nodes for type: 'drug'
  -> Initialized 4159 nodes for type: 'gene/protein'

No explicit .pt/.npy files found. Initializing dummy feature tensors (dim=768) so graph is ready for GNN layers.
  -> Assigned dummy features for 'drug': shape [7801, 768]
  -> Assigned dummy features for 'gene/protein': shape [4159, 768]

=== Graph Construction Complete ===
HeteroData(
  drug={
    num_nodes=7801,
    x=[7801, 768],
  },
  gene/protein={
    num_nodes=4159,
    x=[4159, 768],
  },
  (drug, DPI, gene/protein)={ edge_index=[2, 21447] }
)
Is graph valid? True


In [6]:
# ==========================================
# ROBUST PARSER FOR PrimeKG++ AND DPI
# ==========================================
print("Processing dataset files...")

# Look for standard, compressed, and parquet tables
all_files = []
for ext in ["*.csv", "*.csv.gz", "*.tsv", "*.parquet"]:
    all_files.extend(list(Path(dataset_dir).rglob(ext)))

edge_dfs = []

for file_path in all_files:
    print(f"\nInspecting: {file_path.name}...")
    try:
        # Handle Parquet vs CSV
        if file_path.suffix == '.parquet':
            df = pd.read_parquet(file_path)
        else:
            # low_memory=False prevents mixed-type inference errors on massive KG files
            sep = '\t' if file_path.suffix == '.tsv' else ','
            df = pd.read_csv(file_path, sep=sep, low_memory=False)

    except Exception as e:
        print(f"  [!] FAILED to read {file_path.name}: {e}")
        continue

    print(f"  -> Columns found: {list(df.columns)[:8]}...") # Print first few columns for debugging

    # Standard PrimeKG columns
    edge_cols_1 = {'x_type', 'x_name', 'relation', 'y_type', 'y_name'}
    # Alternative naming conventions (just in case they renamed them in PrimeKG++)
    edge_cols_2 = {'source_type', 'source_name', 'relation', 'target_type', 'target_name'}

    if edge_cols_1.issubset(df.columns):
        print(f"  [SUCCESS] Found standard PrimeKG/DPI edges in: {file_path.name}")
        edge_dfs.append(df[list(edge_cols_1)]) # Keep only what we need to save RAM

    elif edge_cols_2.issubset(df.columns):
        print(f"  [SUCCESS] Found alternative edges in: {file_path.name}")
        # Rename them to match our script's logic
        df.rename(columns={'source_type': 'x_type', 'source_name': 'x_name',
                           'target_type': 'y_type', 'target_name': 'y_name'}, inplace=True)
        edge_dfs.append(df[list(edge_cols_1)])

    else:
        print(f"  [-] Treated as node features/metadata (edge columns not found).")

# ==========================================
# MERGE AND BUILD HETERODATA
# ==========================================
if edge_dfs:
    print("\nConstructing heterogeneous graph edges...")
    full_edges_df = pd.concat(edge_dfs, ignore_index=True)

    # Standardize string capitalization just in case 'Disease' and 'disease' appear
    full_edges_df['x_type'] = full_edges_df['x_type'].str.lower().str.strip()
    full_edges_df['y_type'] = full_edges_df['y_type'].str.lower().str.strip()

    grouped_edges = full_edges_df.groupby(['x_type', 'relation', 'y_type'])

    for (x_type, relation, y_type), group in grouped_edges:
        src_nodes = [get_node_id(x_type, str(name)) for name in group['x_name']]
        dst_nodes = [get_node_id(y_type, str(name)) for name in group['y_name']]

        edge_index = torch.tensor([src_nodes, dst_nodes], dtype=torch.long)
        graph[(x_type, relation, y_type)].edge_index = edge_index
        print(f"  -> Added {len(src_nodes)} edges for: ({x_type}, {relation}, {y_type})")
else:
    print("\n[!] CRITICAL: No edge files were successfully loaded!")

Processing dataset files...

Inspecting: protein_aminoacid_sequence.csv...
  -> Columns found: ['Unnamed: 0', 'protein_id', 'protein_name', 'protein_seq', 'fasta_id', 'fasta_description', 'ncbi_summary']...
  [-] Treated as node features/metadata (edge columns not found).

Inspecting: disease_feature_base.csv...
  -> Columns found: ['Unnamed: 0', 'node_index', 'mondo_id', 'mondo_name', 'group_id_bert', 'group_name_bert', 'mondo_definition', 'umls_description']...
  [-] Treated as node features/metadata (edge columns not found).

Inspecting: drug_feature_base.csv...
  -> Columns found: ['Unnamed: 0', 'node_index', 'description', 'half_life', 'indication', 'mechanism_of_action', 'protein_binding', 'pharmacodynamics']...
  [-] Treated as node features/metadata (edge columns not found).

Inspecting: protein_dna_sequence.csv...
  -> Columns found: ['Unnamed: 0', 'protein_id', 'protein_name', 'protein_seq', 'fasta_id', 'fasta_description', 'ncbi_summary']...
  [-] Treated as node features/me

In [2]:
# ==========================================
# 1. INSTALL DEPENDENCIES
# ==========================================
!pip install -q huggingface_hub torch torch-geometric pandas numpy

import os
import pandas as pd
import numpy as np
import torch
import pickle
from torch_geometric.data import HeteroData
from huggingface_hub import snapshot_download
from pathlib import Path

# ==========================================
# 2. DOWNLOAD DATASET
# ==========================================
print("Downloading BioMedKG dataset...")
dataset_dir = "./BioMedKG_data"
snapshot_download(repo_id="tienda02/BioMedKG", repo_type="dataset", local_dir=dataset_dir)

# ==========================================
# 3. INITIALIZE GRAPH & LOAD EDGES
# ==========================================
graph = HeteroData()
node_mappings = {}
node_counters = {}

def get_node_id(node_type, node_name):
    if node_type not in node_mappings:
        node_mappings[node_type] = {}
        node_counters[node_type] = 0
    if node_name not in node_mappings[node_type]:
        node_mappings[node_type][node_name] = node_counters[node_type]
        node_counters[node_type] += 1
    return node_mappings[node_type][node_name]

# Load the only edge file in the repo
edge_file = Path(dataset_dir) / "dpi" / "dpi_benchmark.csv"
if edge_file.exists():
    print(f"\nLoading edges from {edge_file.name}...")
    df = pd.read_csv(edge_file, low_memory=False)

    grouped_edges = df.groupby(['x_type', 'relation', 'y_type'])
    for (x_type, relation, y_type), group in grouped_edges:
        src_nodes = [get_node_id(x_type, str(name)) for name in group['x_name']]
        dst_nodes = [get_node_id(y_type, str(name)) for name in group['y_name']]

        edge_index = torch.tensor([src_nodes, dst_nodes], dtype=torch.long)
        graph[(x_type, relation, y_type)].edge_index = edge_index
        print(f"  -> Added {len(src_nodes)} edges for: ({x_type}, {relation}, {y_type})")

# Explicitly set num_nodes for PyG
for node_type, count in node_counters.items():
    graph[node_type].num_nodes = count
    print(f"  -> Tracked {count} unique nodes for '{node_type}'")

# ==========================================
# 4. LOAD EMBEDDED FEATURES FROM PICKLES
# ==========================================
print("\nLoading pre-trained multimodal embeddings...")

# Select the embedding file you want to use (e.g., dpi_modality_lm.pickle or a GCL one)
# Change this path to test different embeddings from the gcl_embed folder
embed_file = Path(dataset_dir) / "embed" / "dpi_modality_lm.pickle"

if embed_file.exists():
    with open(embed_file, 'rb') as f:
        embeddings = pickle.load(f)

    # Typically, these pickles contain a dictionary mapping node types to numpy arrays or tensors
    # Format usually looks like: {'drug': array([N, dim]), 'gene/protein': array([M, dim])}

    if isinstance(embeddings, dict):
        for node_type, feat_matrix in embeddings.items():
            # Standardize names to match the CSV (e.g., if pickle uses 'protein' but CSV uses 'gene/protein')
            target_type = 'gene/protein' if 'protein' in node_type or 'gene' in node_type else node_type

            if target_type in graph.node_types:
                # Convert to torch tensor
                tensor_feat = torch.tensor(feat_matrix, dtype=torch.float32) if not isinstance(feat_matrix, torch.Tensor) else feat_matrix.float()
                graph[target_type].x = tensor_feat
                print(f"  [SUCCESS] Attached {tensor_feat.shape} features to '{target_type}'")
            else:
                print(f"  [-] Skipped '{node_type}' from pickle (not found in edge lists).")
    else:
        print("  [!] Pickle structure is not a dictionary. Please inspect `type(embeddings)`.")
else:
    print(f"  [!] Could not find {embed_file}")

# ==========================================
# 5. VALIDATE
# ==========================================
print("\n=== Graph Construction Complete ===")
print(graph)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 23.6 MB/s eta 0:00:00


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 20 files:   0%|          | 0/20 [00:00<?, ?it/s]


Loading edges from dpi_benchmark.csv...
  -> Added 21447 edges for: (drug, DPI, gene/protein)
  -> Tracked 7801 unique nodes for 'drug'
  -> Tracked 4159 unique nodes for 'gene/protein'

Loading pre-trained multimodal embeddings...


/tmp/ipykernel_3259/1891672883.py:69: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  embeddings = pickle.load(f)


  [-] Skipped 'ftsI' from pickle (not found in edge lists).
  [-] Skipped 'HDC' from pickle (not found in edge lists).
  [-] Skipped 'GLS2' from pickle (not found in edge lists).
  [-] Skipped 'F13A1' from pickle (not found in edge lists).
  [-] Skipped 'NOS2' from pickle (not found in edge lists).
  [-] Skipped 'HSD17B2' from pickle (not found in edge lists).
  [-] Skipped 'NNT' from pickle (not found in edge lists).
  [-] Skipped 'ADH5' from pickle (not found in edge lists).
  [-] Skipped 'AMT' from pickle (not found in edge lists).
  [-] Skipped 'IDH3A' from pickle (not found in edge lists).
  [-] Skipped 'CACNA1I' from pickle (not found in edge lists).
  [-] Skipped 'ADORA1' from pickle (not found in edge lists).
  [-] Skipped 'ABL1' from pickle (not found in edge lists).
  [-] Skipped 'FCER1A' from pickle (not found in edge lists).
  [-] Skipped 'F8' from pickle (not found in edge lists).
  [-] Skipped 'PTGS1' from pickle (not found in edge lists).
  [-] Skipped 'ADRBK2' from pick

TypeError: argument of type 'float' is not iterable

In [8]:
# ==========================================
# 1. LOAD REQUIRED LIBRARIES
# ==========================================
import pandas as pd
import numpy as np
import torch
import pickle
from torch_geometric.data import HeteroData
from pathlib import Path

dataset_dir = Path("./BioMedKG_data")
graph = HeteroData()

# ==========================================
# 2. LOAD PRIME-KG EMBEDDINGS (Fixing the TypeError)
# ==========================================
# Switch to the PrimeKG pickle which contains the diseases and proteins
embed_file = dataset_dir / "embed" / "primekg_modality_lm.pickle"

print("Loading PrimeKG multimodal embeddings...")
with open(embed_file, 'rb') as f:
    embeddings = pickle.load(f)

# ==========================================
# 3. IDENTIFY DISEASES VS PROTEINS
# ==========================================
print("Mapping keys to node types...")
# Read the metadata CSVs to find the identifiers
dis_df = pd.read_csv(dataset_dir / "modalities" / "disease_feature_base.csv", low_memory=False)
prot_df = pd.read_csv(dataset_dir / "modalities" / "protein_aminoacid_sequence.csv", low_memory=False)

# Flatten all string columns into a fast-lookup set to match against pickle keys
prot_ids = set(prot_df.select_dtypes(include=['object']).values.flatten())
dis_ids = set(dis_df.select_dtypes(include=['object']).values.flatten())

# Dictionaries to store the continuous integer IDs required by PyG
node_mappings = {'gene/protein': {}, 'disease': {}}
features_list = {'gene/protein': [], 'disease': []}

for key, vector in embeddings.items():
    # Fix for TypeError: Skip any keys that are floats (like NaN)
    if not isinstance(key, str):
        continue

    # Check which node type this key belongs to
    if key in prot_ids:
        node_mappings['gene/protein'][key] = len(node_mappings['gene/protein'])
        features_list['gene/protein'].append(vector)

    elif key in dis_ids:
        node_mappings['disease'][key] = len(node_mappings['disease'])
        features_list['disease'].append(vector)

# ==========================================
# 4. BUILD HETERODATA GRAPH NODES
# ==========================================
print("Populating PyG Graph Nodes...")
for n_type in ['gene/protein', 'disease']:
    if features_list[n_type]:
        # Convert list of arrays to a single PyTorch tensor matrix
        tensor = torch.tensor(np.array(features_list[n_type]), dtype=torch.float32)
        graph[n_type].x = tensor
        graph[n_type].num_nodes = tensor.shape[0]
        print(f"  [SUCCESS] Loaded {tensor.shape[0]} nodes with {tensor.shape[1]}D features for '{n_type}'.")

print("\n=== Graph Nodes Initialized ===")
print(graph)

Loading PrimeKG multimodal embeddings...


/tmp/ipykernel_2020/648548176.py:22: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  embeddings = pickle.load(f)


Mapping keys to node types...
Populating PyG Graph Nodes...
  [SUCCESS] Loaded 20013 nodes with 2D features for 'gene/protein'.
  [SUCCESS] Loaded 22205 nodes with 2D features for 'disease'.

=== Graph Nodes Initialized ===
HeteroData(
  gene/protein={
    x=[20013, 2, 768],
    num_nodes=20013,
  },
  disease={
    x=[22205, 2, 768],
    num_nodes=22205,
  }
)


In [3]:
import pandas as pd
import numpy as np
import pickle
from pathlib import Path
from IPython.display import display

dataset_dir = Path("./BioMedKG_data")

# ==========================================
# 1. LOAD IDENTIFIERS TO CLASSIFY NODES
# ==========================================
print("Loading node identifiers for classification...")
dis_df = pd.read_csv(dataset_dir / "modalities" / "disease_feature_base.csv", low_memory=False)
prot_df = pd.read_csv(dataset_dir / "modalities" / "protein_aminoacid_sequence.csv", low_memory=False)

prot_ids = set(prot_df.select_dtypes(include=['object']).values.flatten())
dis_ids = set(dis_df.select_dtypes(include=['object']).values.flatten())

# ==========================================
# 2. DEFINE THE 4 VERSIONS TO COMPARE
# ==========================================
# We use the 'attention' fusion variant for the GCL models as a baseline
files_to_compare = {
    "Base LM": dataset_dir / "embed" / "primekg_modality_lm.pickle",
    "GCL - DGI": dataset_dir / "gcl_embed" / "dgi_attention.pickle",
    "GCL - GRACE": dataset_dir / "gcl_embed" / "grace_attention.pickle",
    "GCL - GGD": dataset_dir / "gcl_embed" / "ggd_attention.pickle"
}

# ==========================================
# 3. ANALYZE AND COMPARE
# ==========================================
print("Analyzing embedding files (this may take a moment)...\n")
results = []

for name, filepath in files_to_compare.items():
    if not filepath.exists():
        print(f"[!] File not found: {filepath}")
        continue

    with open(filepath, 'rb') as f:
        data = pickle.load(f)

    disease_count = 0
    protein_count = 0
    other_count = 0
    example_shape = None

    for key, vector in data.items():
        # Skip invalid/NaN keys
        if not isinstance(key, str):
            continue

        # Capture the shape of the first valid vector we find
        if example_shape is None:
            example_shape = tuple(np.array(vector).shape)

        # Classify the node
        if key in dis_ids:
            disease_count += 1
        elif key in prot_ids:
            protein_count += 1
        else:
            other_count += 1

    results.append({
        "Embedding Version": name,
        "Feature Shape": str(example_shape),
        "Disease Nodes": disease_count,
        "Protein Nodes": protein_count,
        "Other Nodes": other_count,
        "Total Valid Nodes": disease_count + protein_count + other_count
    })

# ==========================================
# 4. DISPLAY RESULTS
# ==========================================
comparison_df = pd.DataFrame(results)
display(comparison_df)

Loading node identifiers for classification...
Analyzing embedding files (this may take a moment)...



/tmp/ipykernel_3259/4192418963.py:42: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  data = pickle.load(f)


,Embedding Version,Feature Shape,Disease Nodes,Protein Nodes,Other Nodes,Total Valid Nodes
0,Base LM,"(2, 768)",22205,20013,15170,57388
1,GCL - DGI,"(1, 256)",17007,17755,4878,39640
2,GCL - GRACE,"(1, 256)",17007,17755,4878,39640
3,GCL - GGD,"(1, 256)",17007,17755,4878,39640
